In [106]:
!pip install pycountry

import pandas as pd
import numpy as np
import requests
import pycountry

In [107]:
def get_wb_data(indicator: str, col_name: str) -> pd.DataFrame:
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator}"
    params = {
        "format": "json",
        "per_page": 20000
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    json_data = response.json()

    # Check API response structure
    if not isinstance(json_data, list) or len(json_data) < 2 or json_data[1] is None:
        raise ValueError(f"No data returned for indicator: {indicator}")

    data = json_data[1]
    df = pd.json_normalize(data)

    # Keep only needed columns
    df = df[["country.value", "date", "value"]].copy()
    df.columns = ["Country", "Year", col_name]

    # Convert year and values
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df[col_name] = pd.to_numeric(df[col_name], errors="coerce")

    # Drop rows with missing country/year
    df = df.dropna(subset=["Country", "Year"])

    # Make year integer
    df["Year"] = df["Year"].astype(int)

    return df


# Fetch datasets
pm25 = get_wb_data("EN.ATM.PM25.MC.M3", "PM25")
life = get_wb_data("SP.DYN.LE00.IN", "LifeExp")
gdp = get_wb_data("NY.GDP.PCAP.PP.CD", "GDP")
health = get_wb_data("SH.XPD.CHEX.PP.CD", "HealthExp")

In [108]:
# merging datasets
df = pm25.merge(life, on=["Country", "Year"], how="inner")
df = df.merge(gdp, on=["Country", "Year"], how="inner")
df = df.merge(health, on=["Country", "Year"], how="inner")

In [109]:
# merged data description
print(df.shape)
print(df.isna().sum())
print(df.describe())
df

(17556, 6)
Country          0
Year             0
PM25          9868
LifeExp        630
GDP           9045
HealthExp    11846
dtype: int64
               Year         PM25       LifeExp            GDP     HealthExp
count  17556.000000  7688.000000  16926.000000    8511.000000   5710.000000
mean    1992.500000    28.390587     64.735082   16308.860758   1251.598438
std       19.050914    16.436362     11.080176   19995.773964   1733.644790
min     1960.000000     4.895181     10.989000     254.388475      6.106393
25%     1976.000000    16.755624     57.661500    3235.851569    174.489841
50%     1992.500000    23.958606     67.091500    8458.841233    554.732148
75%     2009.000000    39.223020     72.914677   21620.474104   1509.649343
max     2025.000000   107.144665     86.372000  180939.439450  13473.193217


,Country,Year,PM25,LifeExp,GDP,HealthExp
0,Africa Eastern and Southern,2025,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,2024,NaN,NaN,4635.788224,NaN
2,Africa Eastern and Southern,2023,NaN,65.146154,4501.601101,238.850278
3,Africa Eastern and Southern,2022,NaN,64.487020,4369.191056,230.995289
4,Africa Eastern and Southern,2021,NaN,62.979999,4028.376504,223.604661
...,...,...,...,...,...,...
17551,Zimbabwe,1964,NaN,55.431000,NaN,NaN
17552,Zimbabwe,1963,NaN,54.942000,NaN,NaN
17553,Zimbabwe,1962,NaN,54.453000,NaN,NaN
17554,Zimbabwe,1961,NaN,53.966000,NaN,NaN


In [110]:
# view yearly variable nulls
missing_pct = df.groupby("Year").apply(lambda x: x.isna().mean() * 100)
pd.set_option('display.max_rows', None)
print(missing_pct)

      Country  Year        PM25    LifeExp         GDP   HealthExp
Year                                                              
1960      0.0   0.0  100.000000    1.12782  100.000000  100.000000
1961      0.0   0.0  100.000000    0.75188  100.000000  100.000000
1962      0.0   0.0  100.000000    0.75188  100.000000  100.000000
1963      0.0   0.0  100.000000    1.12782  100.000000  100.000000
1964      0.0   0.0  100.000000    1.12782  100.000000  100.000000
1965      0.0   0.0  100.000000    1.12782  100.000000  100.000000
1966      0.0   0.0  100.000000    0.75188  100.000000  100.000000
1967      0.0   0.0  100.000000    0.75188  100.000000  100.000000
1968      0.0   0.0  100.000000    0.75188  100.000000  100.000000
1969      0.0   0.0  100.000000    0.75188  100.000000  100.000000
1970      0.0   0.0  100.000000    0.75188  100.000000  100.000000
1971      0.0   0.0  100.000000    0.75188  100.000000  100.000000
1972      0.0   0.0  100.000000    0.75188  100.000000  100.00

/tmp/ipykernel_13576/1638141631.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_pct = df.groupby("Year").apply(lambda x: x.isna().mean() * 100)


In [111]:
pd.reset_option('display.max_rows')

In [112]:
# restrict year from 2000-2020
df = df[(df["Year"].between(2000, 2020))].dropna()

# drop duplicates
df = df.drop_duplicates(subset=["Country", "Year"]).reset_index(drop=True)

# drop index
df = df.drop(columns=["index"], errors="ignore")

df

,Country,Year,PM25,LifeExp,GDP,HealthExp
0,Africa Eastern and Southern,2020,26.755686,63.766484,3708.094649,210.803136
1,Africa Eastern and Southern,2019,26.248200,63.857261,3801.413183,207.288269
2,Africa Eastern and Southern,2018,27.404295,63.330691,3723.216423,198.891050
3,Africa Eastern and Southern,2017,27.311377,62.591275,3837.726375,205.482092
4,Africa Eastern and Southern,2016,28.877274,62.167981,3634.441673,214.076293
...,...,...,...,...,...,...
4861,Zimbabwe,2014,23.429253,58.106000,3903.427977,203.300574
4862,Zimbabwe,2013,22.598401,56.842000,3783.946337,173.586892
4863,Zimbabwe,2012,23.539528,55.386000,3472.485720,156.086614
4864,Zimbabwe,2011,23.616283,53.911000,3047.317089,161.209011


In [113]:
# check for unnecessary aggregates
df["Country"].unique()

array(['Africa Eastern and Southern', 'Africa Western and Central',
       'Arab World', 'Caribbean small states',
       'Central Europe and the Baltics', 'Early-demographic dividend',
       'East Asia & Pacific',
       'East Asia & Pacific (excluding high income)',
       'East Asia & Pacific (IDA & IBRD countries)', 'Euro area',
       'Europe & Central Asia',
       'Europe & Central Asia (excluding high income)',
       'Europe & Central Asia (IDA & IBRD countries)', 'European Union',
       'Fragile and conflict affected situations',
       'Heavily indebted poor countries (HIPC)', 'High income',
       'IBRD only', 'IDA & IBRD total', 'IDA blend', 'IDA only',
       'IDA total', 'Late-demographic dividend',
       'Latin America & Caribbean',
       'Latin America & Caribbean (excluding high income)',
       'Latin America & the Caribbean (IDA & IBRD countries)',
       'Least developed countries: UN classification',
       'Low & middle income', 'Low income', 'Lower middle in

In [114]:
valid_countries = set([c.name for c in pycountry.countries])

# Keep only real countries
df = df[df["Country"].isin(valid_countries)].reset_index(drop=True)
df

,Country,Year,PM25,LifeExp,GDP,HealthExp
0,Afghanistan,2020,46.087094,61.454,2561.981761,401.163333
1,Afghanistan,2019,58.330872,62.941,2583.485332,383.164896
2,Afghanistan,2018,67.227177,62.443,2432.276701,345.587947
3,Afghanistan,2017,65.862347,62.406,2335.795862,294.796437
4,Afghanistan,2016,72.765910,62.646,2213.181441,261.566867
...,...,...,...,...,...,...
3411,Zimbabwe,2014,23.429253,58.106,3903.427977,203.300574
3412,Zimbabwe,2013,22.598401,56.842,3783.946337,173.586892
3413,Zimbabwe,2012,23.539528,55.386,3472.485720,156.086614
3414,Zimbabwe,2011,23.616283,53.911,3047.317089,161.209011


In [115]:
# TABLE 1
# cleaned data description
print(df.shape)
print(df.isna().sum())
print(df.describe().round(2))

(3416, 6)
Country      0
Year         0
PM25         0
LifeExp      0
GDP          0
HealthExp    0
dtype: int64
          Year     PM25  LifeExp        GDP  HealthExp
count  3416.00  3416.00  3416.00    3416.00    3416.00
mean   2010.06    26.62    70.15   17988.66    1196.89
std       6.05    17.19     8.97   20755.23    1516.70
min    2000.00     4.90    14.66     471.97      10.11
25%    2005.00    14.45    64.35    3601.22     170.50
50%    2010.00    21.70    71.77   10287.08     583.60
75%    2015.00    32.44    76.98   25487.57    1608.38
max    2020.00   107.14    85.26  180939.44   11648.50


In [116]:
# positive value check for log-transformation
df = df[(df["PM25"] > 0) & (df["GDP"] > 0) & (df["HealthExp"] > 0)]

In [117]:
# log transformed dataset (skewness/outlier treatment)
df["log_PM25"] = np.log(df["PM25"])
df["log_GDP"] = np.log(df["GDP"])
df["log_HealthExp"] = np.log(df["HealthExp"])
df

,Country,Year,PM25,LifeExp,GDP,HealthExp,log_PM25,log_GDP,log_HealthExp
0,Afghanistan,2020,46.087094,61.454,2561.981761,401.163333,3.830533,7.848536,5.994369
1,Afghanistan,2019,58.330872,62.941,2583.485332,383.164896,4.066131,7.856895,5.948465
2,Afghanistan,2018,67.227177,62.443,2432.276701,345.587947,4.208078,7.796583,5.845247
3,Afghanistan,2017,65.862347,62.406,2335.795862,294.796437,4.187567,7.756108,5.686285
4,Afghanistan,2016,72.765910,62.646,2213.181441,261.566867,4.287248,7.702186,5.566690
...,...,...,...,...,...,...,...,...,...
3411,Zimbabwe,2014,23.429253,58.106,3903.427977,203.300574,3.153985,8.269610,5.314686
3412,Zimbabwe,2013,22.598401,56.842,3783.946337,173.586892,3.117879,8.238523,5.156678
3413,Zimbabwe,2012,23.539528,55.386,3472.485720,156.086614,3.158681,8.152626,5.050411
3414,Zimbabwe,2011,23.616283,53.911,3047.317089,161.209011,3.161936,8.022017,5.082702


In [118]:
df.to_excel("195_final_cleaned_dataset.xlsx", index=False)